# 04 — Model driver circuit specialization

## Purpose

Estimate driver circuit specialization with transparent ordinary least squares regression.

First estimate expected performance from normalized starting position and constructor-season effects. Then model the remaining performance residual separately across speed, technical, downforce, and overtaking circuit characteristics.

This notebook measures specialization, not all-time driver greatness. It uses no machine learning, Bayesian shrinkage, hierarchical modeling, or confidence-based score adjustment.

## Inputs

- `data/processed/driver_race_with_circuit_features.csv`

The modeling population includes drivers who started and were not disqualified. DNFs remain in the main analysis to avoid conditioning on successful finishes. Drivers require at least 50 eligible career starts for specialization models.

## Outputs

- `data/outputs/driver_track_model_results.csv`
- `data/outputs/driver_specialization_scores.csv`
- `data/outputs/model_coefficients.csv`
- `data/outputs/model_summary_metrics.csv`
- `data/outputs/notebook_04_validation_report.json`

## Imports

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

## Configuration

In [2]:
START_YEAR = 2004
END_YEAR = 2025
MINIMUM_CAREER_STARTS = 50

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

INPUT_PATH = PROCESSED_DIR / "driver_race_with_circuit_features.csv"
OUTPUT_PATHS = {
    "driver_track_model_results": OUTPUT_DIR / "driver_track_model_results.csv",
    "driver_specialization_scores": OUTPUT_DIR / "driver_specialization_scores.csv",
    "model_coefficients": OUTPUT_DIR / "model_coefficients.csv",
    "model_summary_metrics": OUTPUT_DIR / "model_summary_metrics.csv",
}
VALIDATION_REPORT_PATH = OUTPUT_DIR / "notebook_04_validation_report.json"

DRIVER_RACE_KEY = ["season", "round", "driver_id"]
BASELINE_FORMULA = (
    "actual_performance ~ starting_position_pct "
    "+ pit_lane_start_flag + C(constructor_season)"
)
TRAIT_COLUMNS = {
    "speed": "speed_zscore",
    "technical": "technical_zscore",
    "downforce": "downforce_zscore",
    "overtaking": "overtaking_zscore",
}
TRAIT_DIRECTIONS = {
    "speed": ("low_speed", "high_speed"),
    "technical": ("low_technical", "high_technical"),
    "downforce": ("low_downforce", "high_downforce"),
    "overtaking": ("overtaking_difficult", "overtaking_friendly"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

In [3]:
def require_file(path: Path) -> None:
    """Raise a readable error when the required input is absent."""
    if not path.exists():
        raise FileNotFoundError(f"Missing Notebook 3 output: {path}")


def filter_scope(frame: pd.DataFrame) -> pd.DataFrame:
    """Restrict rows to the configured inclusive season range."""
    seasons = pd.to_numeric(frame["season"], errors="coerce")
    return frame.loc[seasons.between(START_YEAR, END_YEAR)].reset_index(drop=True)


def save_csv(frame: pd.DataFrame, path: Path) -> None:
    """Write a CSV atomically so a partial file is never treated as final."""
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def report_check(name: str, passed: bool, details: Any) -> Dict[str, Any]:
    """Create a JSON-serializable validation record."""
    return {"check": name, "passed": bool(passed), "details": details}


def normalized_csv(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize values through strings for a CSV round-trip comparison."""
    return frame.reset_index(drop=True).astype("string").fillna("").astype(str)


def coefficient_frame(model: Any, model_name: str) -> pd.DataFrame:
    """Return a long-form coefficient table for one fitted OLS model."""
    confidence = model.conf_int()
    return pd.DataFrame(
        {
            "model_name": model_name,
            "term": model.params.index,
            "coefficient": model.params.values,
            "standard_error": model.bse.values,
            "t_statistic": model.tvalues.values,
            "p_value": model.pvalues.values,
            "confidence_interval_low": confidence.iloc[:, 0].values,
            "confidence_interval_high": confidence.iloc[:, 1].values,
        }
    )


def summary_row(
    model: Any,
    model_name: str,
    target: str,
    formula: str,
    driver_count: int,
) -> Dict[str, Any]:
    """Create one compact model-summary record."""
    residuals = np.asarray(model.resid, dtype=float)
    return {
        "model_name": model_name,
        "target": target,
        "formula": formula,
        "observations": int(model.nobs),
        "drivers": int(driver_count),
        "parameters": int(len(model.params)),
        "r_squared": float(model.rsquared),
        "adjusted_r_squared": float(model.rsquared_adj),
        "rmse": float(np.sqrt(np.mean(np.square(residuals)))),
    }


def specialization_formula(trait_column: str) -> str:
    """Return a no-reference-driver formula with one intercept and slope per driver."""
    return (
        "baseline_residual ~ 0 + C(driver_id) "
        f"+ C(driver_id):{trait_column}"
    )

## Processing

In [4]:
require_file(INPUT_PATH)
model_source = filter_scope(pd.read_csv(INPUT_PATH))

# Performance is field-relative and oriented so higher values are better.
model_source["actual_performance"] = 1.0 - model_source["finish_position_pct"]
model_source["constructor_season"] = (
    model_source["season"].astype(int).astype(str)
    + "_"
    + model_source["constructor_id"].astype(str)
)
model_source["pit_lane_start_flag"] = (
    model_source["grid_zero_flag"].eq(1)
    & model_source["did_start"].eq(1)
).astype(int)
model_source["capped_effective_grid_position"] = np.minimum(
    model_source["effective_grid_position"],
    model_source["field_size"],
)
model_source["starting_position_pct"] = (
    (model_source["capped_effective_grid_position"] - 1)
    / (model_source["field_size"] - 1)
)

baseline_model_rows = model_source["model_eligible_row"].eq(1)
baseline_data = model_source.loc[baseline_model_rows].copy()
baseline_model = smf.ols(BASELINE_FORMULA, data=baseline_data).fit()
baseline_data["expected_performance"] = baseline_model.predict(baseline_data)
baseline_data["baseline_residual"] = (
    baseline_data["actual_performance"] - baseline_data["expected_performance"]
)

# Career eligibility is based on eligible starts, not successful finishes.
career_starts = baseline_data.groupby("driver_id").size().rename("career_starts")
eligible_driver_ids = career_starts.loc[
    career_starts.ge(MINIMUM_CAREER_STARTS)
].index
specialization_data = baseline_data.loc[
    baseline_data["driver_id"].isin(eligible_driver_ids)
].copy()

# Row-level output retains every source row and marks model participation explicitly.
track_output_columns = [
    *DRIVER_RACE_KEY, "race_name", "circuit_id", "circuit_name",
    "driver_full_name", "constructor_id", "constructor_name",
    "constructor_season", "position", "field_size", "finish_status_group",
    "model_eligible_row", "actual_performance", "effective_grid_position",
    "capped_effective_grid_position", "starting_position_pct",
    "pit_lane_start_flag", "speed_zscore", "technical_zscore",
    "downforce_zscore", "overtaking_zscore",
]
driver_track_model_results = model_source[track_output_columns].copy()
driver_track_model_results["baseline_model_row"] = baseline_model_rows.astype(int)
driver_track_model_results["specialization_model_row"] = (
    driver_track_model_results["baseline_model_row"].eq(1)
    & driver_track_model_results["driver_id"].isin(eligible_driver_ids)
).astype(int)
driver_track_model_results = driver_track_model_results.merge(
    baseline_data[
        DRIVER_RACE_KEY + ["expected_performance", "baseline_residual"]
    ],
    on=DRIVER_RACE_KEY,
    how="left",
    validate="one_to_one",
)

model_coefficients_parts = [
    coefficient_frame(baseline_model, "baseline_expected_performance")
]
model_summary_rows = [
    summary_row(
        baseline_model,
        "baseline_expected_performance",
        "actual_performance",
        BASELINE_FORMULA,
        baseline_data["driver_id"].nunique(),
    )
]

driver_names = (
    specialization_data[
        ["driver_id", "driver_full_name"]
    ].drop_duplicates("driver_id")
)
driver_specialization_scores = driver_names.merge(
    career_starts.rename_axis("driver_id").reset_index(),
    on="driver_id",
    how="left",
    validate="one_to_one",
)
driver_specialization_scores["races_used"] = (
    specialization_data.groupby("driver_id").size()
    .reindex(driver_specialization_scores["driver_id"]).to_numpy()
)
driver_specialization_scores["circuits_raced"] = (
    specialization_data.groupby("driver_id")["circuit_id"].nunique()
    .reindex(driver_specialization_scores["driver_id"]).to_numpy()
)
driver_specialization_scores["mean_baseline_residual"] = (
    specialization_data.groupby("driver_id")["baseline_residual"].mean()
    .reindex(driver_specialization_scores["driver_id"]).to_numpy()
)

specialization_models = {}
for trait_name, trait_column in TRAIT_COLUMNS.items():
    formula = specialization_formula(trait_column)
    model = smf.ols(formula, data=specialization_data).fit()
    specialization_models[trait_name] = model
    model_coefficients_parts.append(
        coefficient_frame(model, f"{trait_name}_specialization")
    )
    model_summary_rows.append(
        summary_row(
            model,
            f"{trait_name}_specialization",
            "baseline_residual",
            formula,
            specialization_data["driver_id"].nunique(),
        )
    )

    intercepts = {}
    slopes = {}
    for driver_id in eligible_driver_ids:
        intercept_term = f"C(driver_id)[{driver_id}]"
        slope_term = f"C(driver_id)[{driver_id}]:{trait_column}"
        intercepts[driver_id] = float(model.params[intercept_term])
        slopes[driver_id] = float(model.params[slope_term])

    driver_specialization_scores[f"{trait_name}_intercept"] = (
        driver_specialization_scores["driver_id"].map(intercepts)
    )
    driver_specialization_scores[f"{trait_name}_slope"] = (
        driver_specialization_scores["driver_id"].map(slopes)
    )
    low_label, high_label = TRAIT_DIRECTIONS[trait_name]
    driver_specialization_scores[f"{trait_name}_direction"] = np.where(
        driver_specialization_scores[f"{trait_name}_slope"].ge(0),
        high_label,
        low_label,
    )
    driver_specialization_scores[f"{trait_name}_high_trait_rank"] = (
        driver_specialization_scores[f"{trait_name}_slope"]
        .rank(method="min", ascending=False)
        .astype("Int64")
    )
    driver_specialization_scores[f"{trait_name}_low_trait_rank"] = (
        driver_specialization_scores[f"{trait_name}_slope"]
        .rank(method="min", ascending=True)
        .astype("Int64")
    )

slope_columns = [f"{trait}_slope" for trait in TRAIT_COLUMNS]
driver_specialization_scores["specialization_magnitude"] = np.sqrt(
    np.mean(
        np.square(driver_specialization_scores[slope_columns].to_numpy()),
        axis=1,
    )
)
driver_specialization_scores["versatility_score"] = (
    -driver_specialization_scores["specialization_magnitude"]
)
driver_specialization_scores["versatility_rank"] = (
    driver_specialization_scores["specialization_magnitude"]
    .rank(method="min", ascending=True)
    .astype("Int64")
)
driver_specialization_scores["overall_outperformance_rank"] = (
    driver_specialization_scores["mean_baseline_residual"]
    .rank(method="min", ascending=False)
    .astype("Int64")
)
driver_specialization_scores = driver_specialization_scores.sort_values(
    ["versatility_rank", "driver_id"]
).reset_index(drop=True)

model_coefficients = pd.concat(
    model_coefficients_parts,
    ignore_index=True,
)
model_summary_metrics = pd.DataFrame(model_summary_rows)
driver_track_model_results = driver_track_model_results.sort_values(
    ["season", "round", "position", "driver_id"]
).reset_index(drop=True)

## Diagnostics

In [5]:
diagnostics = pd.DataFrame(
    {
        "metric": [
            "source rows", "baseline observations", "eligible drivers",
            "specialization observations", "baseline R-squared",
            "baseline RMSE", "mean baseline residual",
        ],
        "value": [
            len(model_source),
            len(baseline_data),
            len(eligible_driver_ids),
            len(specialization_data),
            baseline_model.rsquared,
            np.sqrt(np.mean(np.square(baseline_model.resid))),
            baseline_data["baseline_residual"].mean(),
        ],
    }
)
display(diagnostics)
display(model_summary_metrics)
display(
    driver_specialization_scores[
        [
            "driver_full_name", "career_starts", "speed_slope",
            "technical_slope", "downforce_slope", "overtaking_slope",
            "specialization_magnitude", "versatility_rank",
        ]
    ].head(15)
)
print(
    "Interpretation note: overtaking slope measures specialization across "
    "overtaking environments, not direct on-track pass counts."
)

,metric,value
0,source rows,9.125000e+03
1,baseline observations,9.054000e+03
2,eligible drivers,5.100000e+01
3,specialization observations,7.697000e+03
4,baseline R-squared,4.555997e-01
5,baseline RMSE,2.224242e-01
6,mean baseline residual,-3.603872e-16


,model_name,target,formula,observations,drivers,parameters,r_squared,adjusted_r_squared,rmse
0,baseline_expected_performance,actual_performance,actual_performance ~ starting_position_pct + p...,9054,111,235,0.455600,0.441155,0.222424
1,speed_specialization,baseline_residual,baseline_residual ~ 0 + C(driver_id) + C(drive...,7697,51,102,0.016539,0.003461,0.227320
2,technical_specialization,baseline_residual,baseline_residual ~ 0 + C(driver_id) + C(drive...,7697,51,102,0.015592,0.002501,0.227429
3,downforce_specialization,baseline_residual,baseline_residual ~ 0 + C(driver_id) + C(drive...,7697,51,102,0.015627,0.002537,0.227425
4,overtaking_specialization,baseline_residual,baseline_residual ~ 0 + C(driver_id) + C(drive...,7697,51,102,0.015579,0.002488,0.227430


,driver_full_name,career_starts,speed_slope,technical_slope,downforce_slope,overtaking_slope,specialization_magnitude,versatility_rank
0,Alexander Albon,128,0.003780,-0.008777,-0.004220,0.002162,0.005334,1
1,Daniel Ricciardo,255,-0.003438,0.006609,0.003277,-0.007880,0.005664,2
2,Lewis Hamilton,377,-0.006204,-0.001850,0.009084,-0.005500,0.006218,3
3,Nicholas Latifi,61,0.002039,0.010170,-0.007056,-0.003345,0.006491,4
4,Yuki Tsunoda,113,0.005715,0.000105,-0.008638,0.012310,0.008044,5
5,Marcus Ericsson,97,-0.009948,0.009482,0.008160,-0.006529,0.008633,6
6,Kevin Magnussen,184,0.007916,-0.014414,-0.005337,-0.000110,0.008645,7
7,Mark Webber,183,-0.005398,0.013507,0.009087,-0.002927,0.008699,8
8,Robert Kubica,98,0.005000,-0.002264,-0.016737,0.000272,0.008808,9
9,Felipe Massa,251,0.014444,0.003191,-0.009305,0.005530,0.009165,10


Interpretation note: overtaking slope measures specialization across overtaking environments, not direct on-track pass counts.


## Save outputs

In [6]:
frames_to_save = {
    "driver_track_model_results": driver_track_model_results,
    "driver_specialization_scores": driver_specialization_scores,
    "model_coefficients": model_coefficients,
    "model_summary_metrics": model_summary_metrics,
}
for name, frame in frames_to_save.items():
    save_csv(frame, OUTPUT_PATHS[name])
    print(
        f"Saved {name}: {len(frame):,} rows to "
        f"{OUTPUT_PATHS[name].relative_to(PROJECT_ROOT)}"
    )

Saved driver_track_model_results: 9,125 rows to data/outputs/driver_track_model_results.csv
Saved driver_specialization_scores: 51 rows to data/outputs/driver_specialization_scores.csv
Saved model_coefficients: 643 rows to data/outputs/model_coefficients.csv
Saved model_summary_metrics: 5 rows to data/outputs/model_summary_metrics.csv


## Validation

In [7]:
expected_driver_count = len(eligible_driver_ids)
specialization_coefficient_counts = {}
for trait_name, trait_column in TRAIT_COLUMNS.items():
    terms = model_coefficients.loc[
        model_coefficients["model_name"].eq(f"{trait_name}_specialization"),
        "term",
    ]
    specialization_coefficient_counts[trait_name] = int(
        terms.str.endswith(f":{trait_column}").sum()
    )

recomputed_magnitude = np.sqrt(
    np.mean(
        np.square(driver_specialization_scores[slope_columns].to_numpy()),
        axis=1,
    )
)
residual_identity_error = np.nanmax(
    np.abs(
        driver_track_model_results.loc[
            driver_track_model_results["baseline_model_row"].eq(1),
            "baseline_residual",
        ]
        - (
            driver_track_model_results.loc[
                driver_track_model_results["baseline_model_row"].eq(1),
                "actual_performance",
            ]
            - driver_track_model_results.loc[
                driver_track_model_results["baseline_model_row"].eq(1),
                "expected_performance",
            ]
        )
    )
)

checks = [
    report_check(
        "baseline population is complete",
        len(baseline_data) == int(model_source["model_eligible_row"].sum()),
        {
            "observations": len(baseline_data),
            "expected": int(model_source["model_eligible_row"].sum()),
        },
    ),
    report_check(
        "minimum-start eligibility is enforced",
        driver_specialization_scores["career_starts"].ge(
            MINIMUM_CAREER_STARTS
        ).all(),
        {
            "drivers": len(driver_specialization_scores),
            "minimum_starts": int(
                driver_specialization_scores["career_starts"].min()
            ),
            "threshold": MINIMUM_CAREER_STARTS,
        },
    ),
    report_check(
        "expected eligible-driver count",
        expected_driver_count == 51,
        expected_driver_count,
    ),
    report_check(
        "baseline residuals average zero",
        abs(float(baseline_data["baseline_residual"].mean())) < 1e-12,
        float(baseline_data["baseline_residual"].mean()),
    ),
    report_check(
        "positive residual means actual exceeds expected",
        residual_identity_error < 1e-12,
        float(residual_identity_error),
    ),
    report_check(
        "all specialization models return one slope per eligible driver",
        all(
            count == expected_driver_count
            for count in specialization_coefficient_counts.values()
        ),
        specialization_coefficient_counts,
    ),
    report_check(
        "driver specialization table has one row per eligible driver",
        len(driver_specialization_scores) == expected_driver_count
        and not driver_specialization_scores["driver_id"].duplicated().any(),
        {
            "rows": len(driver_specialization_scores),
            "duplicate_drivers": int(
                driver_specialization_scores["driver_id"].duplicated().sum()
            ),
        },
    ),
    report_check(
        "all driver-trait slopes are present",
        not driver_specialization_scores[slope_columns].isna().any().any(),
        driver_specialization_scores[slope_columns].isna().sum().to_dict(),
    ),
    report_check(
        "versatility magnitude reproduces from raw slopes",
        np.allclose(
            driver_specialization_scores["specialization_magnitude"],
            recomputed_magnitude,
            rtol=0,
            atol=1e-12,
        ),
        float(
            np.max(
                np.abs(
                    driver_specialization_scores[
                        "specialization_magnitude"
                    ].to_numpy()
                    - recomputed_magnitude
                )
            )
        ),
    ),
    report_check(
        "versatility score is unadjusted negative magnitude",
        np.allclose(
            driver_specialization_scores["versatility_score"],
            -driver_specialization_scores["specialization_magnitude"],
            rtol=0,
            atol=1e-12,
        ),
        None,
    ),
    report_check(
        "model summary contains baseline plus four specialization models",
        len(model_summary_metrics) == 5
        and model_summary_metrics["model_name"].nunique() == 5,
        model_summary_metrics["model_name"].tolist(),
    ),
    report_check(
        "row-level output preserves all source rows",
        len(driver_track_model_results) == len(model_source)
        and not driver_track_model_results.duplicated(
            DRIVER_RACE_KEY
        ).any(),
        {
            "source_rows": len(model_source),
            "output_rows": len(driver_track_model_results),
        },
    ),
    report_check(
        "specialization models use standardized circuit traits",
        all(column.endswith("_zscore") for column in TRAIT_COLUMNS.values()),
        TRAIT_COLUMNS,
    ),
    report_check(
        "baseline uses constructor-season effects",
        "C(constructor_season)" in BASELINE_FORMULA,
        BASELINE_FORMULA,
    ),
]

validation_report = {
    "notebook": "04_driver_track_model",
    "scope": {"start_year": START_YEAR, "end_year": END_YEAR},
    "minimum_career_starts": MINIMUM_CAREER_STARTS,
    "status": "PASS" if all(check["passed"] for check in checks) else "FAIL",
    "checks": checks,
}
display(pd.DataFrame(checks))
if validation_report["status"] != "PASS":
    failed = [check for check in checks if not check["passed"]]
    raise AssertionError(f"Validation failed: {failed}")
print("PASS")

,check,passed,details
0,baseline population is complete,True,"{'observations': 9054, 'expected': 9054}"
1,minimum-start eligibility is enforced,True,"{'drivers': 51, 'minimum_starts': 50, 'thresho..."
2,expected eligible-driver count,True,51
3,baseline residuals average zero,True,-0.0
4,positive residual means actual exceeds expected,True,0.0
5,all specialization models return one slope per...,True,"{'speed': 51, 'technical': 51, 'downforce': 51..."
6,driver specialization table has one row per el...,True,"{'rows': 51, 'duplicate_drivers': 0}"
7,all driver-trait slopes are present,True,"{'speed_slope': 0, 'technical_slope': 0, 'down..."
8,versatility magnitude reproduces from raw slopes,True,0.0
9,versatility score is unadjusted negative magni...,True,None


PASS


## Supercheck

In [8]:
superchecks = []
for name, frame in frames_to_save.items():
    path = OUTPUT_PATHS[name]
    saved_frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    equal = normalized_csv(frame).equals(normalized_csv(saved_frame))
    superchecks.append(
        report_check(f"{name} file exists", path.exists(), str(path))
    )
    superchecks.append(
        report_check(
            f"{name} saved data equals memory",
            equal,
            {"rows": len(saved_frame), "columns": len(saved_frame.columns)},
        )
    )

validation_report["superchecks"] = superchecks
validation_report["status"] = (
    "PASS" if all(check["passed"] for check in checks + superchecks) else "FAIL"
)
serializable_report = json.loads(json.dumps(validation_report, default=str))
temporary_report_path = VALIDATION_REPORT_PATH.with_suffix(".json.tmp")
with temporary_report_path.open("w", encoding="utf-8") as handle:
    json.dump(serializable_report, handle, indent=2)
temporary_report_path.replace(VALIDATION_REPORT_PATH)

saved_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding="utf-8"))
report_checks = [
    report_check(
        "validation report file exists",
        VALIDATION_REPORT_PATH.exists(),
        str(VALIDATION_REPORT_PATH),
    ),
    report_check(
        "saved validation report equals memory",
        saved_report == serializable_report,
        None,
    ),
]
display(pd.DataFrame(superchecks + report_checks))
if validation_report["status"] != "PASS" or not all(
    check["passed"] for check in report_checks
):
    raise AssertionError("Supercheck failed")
print("PASS")

,check,passed,details
0,driver_track_model_results file exists,True,
1,driver_track_model_results saved data equals m...,True,"{'rows': 9125, 'columns': 27}"
2,driver_specialization_scores file exists,True,
3,driver_specialization_scores saved data equals...,True,"{'rows': 51, 'columns': 30}"
4,model_coefficients file exists,True,
5,model_coefficients saved data equals memory,True,"{'rows': 643, 'columns': 8}"
6,model_summary_metrics file exists,True,
7,model_summary_metrics saved data equals memory,True,"{'rows': 5, 'columns': 9}"
8,validation report file exists,True,
9,saved validation report equals memory,True,None


PASS
